In [1]:
!pip install -q pyannote.audio pyannote.metrics

import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 17.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 79.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!ls /kaggle/input/          # confirm both slugs

datasets  notebooks


In [4]:
import pathlib, shutil, os

ROOT = pathlib.Path("/kaggle/input")

# Find the directory that actually contains the scripts, wherever it landed.
CODE = next(p.parent for p in ROOT.rglob("stage3_diarize.py"))
# Find the directory that actually contains the WAVs.
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
REF = next(p.parent for p in ROOT.rglob("clip_meta.csv"))
WORK = "/kaggle/working/data"

print("CODE  =", CODE)
print("AUDIO =", AUDIO, f"({len(list(AUDIO.glob('*.wav')))} wavs)")
print("REF   =", REF, f"({len(list(REF.rglob('*')))} files)")

CODE  = /kaggle/input/notebooks/ritankarmondal/sarvam-initial
AUDIO = /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav (99 wavs)
REF   = /kaggle/input/notebooks/ritankarmondal/sarvam-initial/data/ref (205 files)


In [5]:
pathlib.Path(WORK).mkdir(parents=True, exist_ok=True)
shutil.copytree(REF, f"{WORK}/ref", dirs_exist_ok=True)
for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")
mf = next(ROOT.rglob("manifest.jsonl"), None)
if mf: shutil.copy(mf, WORK)

print("ref files:", len(list(pathlib.Path(f'{WORK}/ref').rglob('*'))))
print("scripts  :", [p.name for p in pathlib.Path('/kaggle/working').glob('*.py')])

ref files: 205
scripts  : ['stage3_score.py', 'stage1_extract.py', 'build_notebooks.py', 'stage3_diarize.py', 'stage2_parse_refs.py']


In [6]:
import shutil, pathlib
PREV = pathlib.Path("/kaggle/input/notebooks/ritankarmondal/sarvam-initial")   # fix the slug
DST  = pathlib.Path("/kaggle/working/data")

assert (PREV / "data").exists(), sorted(p.name for p in PREV.iterdir())
DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV / "data", DST, dirs_exist_ok=True)

print("ref  :", len(list((DST / "ref/rttm").glob("*.rttm"))))
print("hyp  :", len(list((DST / "hyp").rglob("*.rttm"))))

ref  : 100
hyp  : 272


In [7]:
!pip install -q "nemo_toolkit[asr]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.8/242.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 83.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# !python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO}

In [ ]:
# !python stage3_diarize.py --system community1 --data data --wav-dir {AUDIO} --limit 5
# !python stage3_diarize.py --system community1 --data data --wav-dir {AUDIO}

In [7]:
import shutil, pathlib
PREV = pathlib.Path("/kaggle/input/notebooks/ritankarmondal/sarvam-initial")   # fix the slug
DST  = pathlib.Path("/kaggle/working/data")

assert (PREV / "data").exists(), sorted(p.name for p in PREV.iterdir())
DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV / "data", DST, dirs_exist_ok=True)

print("ref  :", len(list((DST / "ref/rttm").glob("*.rttm"))))
print("hyp  :", len(list((DST / "hyp").rglob("*.rttm"))))

ref  : 100
hyp  : 272


In [9]:
!python stage3_score.py --data data --systems pyannote31 sortformer

[ok] scored pyannote31: 99 clips
[ok] scored sortformer: 99 clips

STAGE 3 -- BASELINE DIARIZATION   (collar=0.0, skip_overlap=False, UEM=full clip)
system             DER    miss      FA    conf     JER  spk acc  spk MAE
------------------------------------------------------------------------------
pyannote31      27.34%  11.59%   5.89%   9.86%  38.14%    72.7%     0.33
sortformer      74.85%  65.14%   2.19%   7.52%  65.29%    43.4%     1.55
------------------------------------------------------------------------------
DER/miss/FA/conf are duration-weighted. Macro (per-clip mean) for contrast:
  pyannote31     DER_macro  29.76%   JER_macro  38.09%   (weighted DER 27.34%)
  sortformer     DER_macro  52.02%   JER_macro  59.57%   (weighted DER 74.85%)
    [!] 25 clip(s) had NO hypothesis (scored as total miss)

------------------------------------------------------------------------------
DER by reference speaker count (duration-weighted within each bucket):
-----------------------------

In [10]:
import pandas as pd
pd.read_csv("/kaggle/working/data/results/diarization_summary.csv")

,system,n_clips,n_hyp_missing,DER,miss,false_alarm,confusion,DER_pyannote_accum,JER_pyannote_accum,DER_macro,JER_macro,spk_count_acc,spk_count_mae,spk_count_bias
0,pyannote31,99,0,0.27338,0.115851,0.05891,0.09862,0.27338,0.381357,0.29759,0.38092,0.727273,0.333333,-0.131313
1,sortformer,99,99,1.00000,1.000000,0.00000,0.00000,1.00000,1.000000,1.00000,1.00000,0.000000,3.515152,-3.515152


In [8]:
import collections
import json
import pathlib

data_dir = pathlib.Path("/kaggle/working/data")
manifest = data_dir / "manifest.jsonl"
hyp_dir = data_dir / "hyp"

rttm_count = len(list(hyp_dir.rglob("*.rttm"))) if hyp_dir.exists() else 0

print(f"Data directory:  {data_dir.exists()}")
print(f"Manifest exists: {manifest.exists()}")
print(f"Hyp RTTM count:  {rttm_count}")

if not manifest.exists():
    print("\n[!] manifest.jsonl not found in /kaggle/working/data/")
else:
    recs = [json.loads(line) for line in manifest.open(encoding="utf-8") if line.strip()]
    status_counts = collections.Counter(r.get("status") for r in recs)
    print(f"\n{len(recs)} records processed: {dict(status_counts)}")

    failed = [(r.get("error") or "Unknown error")[:160] for r in recs if r.get("status") != "ok"]
    if failed:
        print("\nTop errors:")
        for err, count in collections.Counter(failed).most_common(5):
            print(f"  x{count}  {err}")

Data directory:  True
Manifest exists: True
Hyp RTTM count:  272

100 records processed: {'ok': 99, 'failed': 1}

Top errors:
  x1  Unknown error


In [10]:
# import json, collections, pathlib

# d = pathlib.Path('/kaggle/working/data/hyp/sortformer')
# print('dir exists:', d.exists())
# if d.exists():
#     print('contents:', sorted(p.name for p in d.iterdir()))
#     print('rttms   :', len(list((d / 'rttm').glob('*.rttm'))) if (d / 'rttm').exists() else 0)

# p = d / 'manifest.jsonl'
# if not p.exists():
#     print('NO MANIFEST -- the sortformer diarize run never started')
# else:
#     recs = [json.loads(l) for l in p.open(encoding='utf-8') if l.strip()]
#     print(len(recs), 'records', collections.Counter(r['status'] for r in recs))
#     errs = collections.Counter((r.get('error') or '')[:200]
#                                for r in recs if r['status'] != 'ok')
#     for e, n in errs.most_common(5):
#         print(f'  x{n}  {e}')

dir exists: True
contents: ['manifest.jsonl', 'rttm']
rttms   : 74
101 records Counter({'ok': 74, 'failed': 27})
  x1  OutOfMemoryError: CUDA out of memory. Tried to allocate 7.77 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.56 GiB is free. Process 23 has 882.00 MiB memory in use. Including non-PyTorch memo
  x1  OutOfMemoryError: CUDA out of memory. Tried to allocate 7.85 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.48 GiB is free. Process 23 has 882.00 MiB memory in use. Including non-PyTorch memo
  x1  OutOfMemoryError: CUDA out of memory. Tried to allocate 7.77 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.58 GiB is free. Process 23 has 882.00 MiB memory in use. Including non-PyTorch memo
  x1  OutOfMemoryError: CUDA out of memory. Tried to allocate 7.85 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.50 GiB is free. Process 23 has 882.00 MiB memory in use. Including non-PyTorch memo
  x1  OutOfMemoryError: CUDA out of memory. Tried to alloca

In [25]:
!pip install -q "nemo_toolkit[asr]"

In [11]:
import torch
from nemo.collections.asr.models import SortformerEncLabelModel

MODEL_ID = "nvidia/diar_sortformer_4spk-v1"

m = SortformerEncLabelModel.from_pretrained(MODEL_ID)
m.eval()
m.to(torch.device("cuda"))

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


diar_sortformer_4spk-v1.nemo:   0%|          | 0.00/493M [00:00<?, ?B/s]

[NeMo W 2026-09-07 17:13:27 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-09-07 17:13:27 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-09-07 17:13:29 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.


SortformerEncLabelModel(
  (preprocessor): AudioToMelSpectrogramPreprocessor(
    (featurizer): FilterbankFeatures()
  )
  (encoder): ConformerEncoder(
    (pre_encode): ConvSubsampling(
      (out): Linear(in_features=2560, out_features=512, bias=True)
      (conv): MaskedConvSequential(
        (0): Conv2d(1, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        (3): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
        (4): ReLU(inplace=True)
        (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        (6): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
        (7): ReLU(inplace=True)
      )
    )
    (pos_enc): RelPositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-17): 18 x ConformerLayer(
        (norm_feed_forward1): LayerNorm((512,), eps=1

In [28]:
# wav = sorted(AUDIO.glob("*.wav"))[0]
# pred = m.diarize(audio=[str(wav)], batch_size=1)

# print("type :", type(pred))
# print("len  :", len(pred))
# inner = pred[0]
# print("inner:", type(inner), len(inner) if hasattr(inner, "__len__") else "-")

# items = inner if isinstance(inner, list) else pred
# for x in items[:3]:
#     print("  ", type(x).__name__, repr(x)[:200])

[NeMo I 2026-09-07 08:40:22 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-09-07 08:40:22 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: session_len_sec,soft_label_thres,num_spks
Diarizing: 1it [00:29, 29.32s/it]

type : <class 'list'>
len  : 1
inner: <class 'list'> 61
   str '0.960 2.480 speaker_0'
   str '5.520 8.480 speaker_0'
   str '8.560 10.480 speaker_0'


In [29]:
!python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO} --limit 5

[env ] system=sortformer  model=nvidia/diar_sortformer_4spk-v1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[warn] sortformer is capped at 4 speakers; clips above that cannot be solved and should be reported separately
[run ] 5 of 5 clips to do (0 already done)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-09-07 08:42:10 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_siz

In [ ]:
# !python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO}

In [12]:
# import inspect
# print(inspect.signature(m.diarize))
# print("---")
# sm = getattr(m, "sortformer_modules", None)
# print([a for a in dir(sm) if not a.startswith("_")] if sm else "no sortformer_modules")
# print("---")
# print((m.diarize.__doc__ or "")[:1500])

(audio: Union[str, List[str], numpy.ndarray, torch.utils.data.dataloader.DataLoader], sample_rate: Optional[int] = None, batch_size: int = 1, include_tensor_outputs: bool = False, postprocessing_yaml: Optional[str] = None, num_workers: int = 0, verbose: bool = True, override_config: Optional[nemo.collections.asr.parts.mixins.diarization.DiarizeConfig] = None) -> Union[List[List[str]], Tuple[List[List[str]], List[torch.Tensor]]]
---
['T_destination', 'add_module', 'apply', 'apply_mask_to_preds', 'as_frozen', 'bfloat16', 'buffers', 'call_super_init', 'causal_attn_rate', 'causal_attn_rc', 'children', 'chunk_left_context', 'chunk_len', 'chunk_right_context', 'compile', 'concat_and_pad', 'concat_embs', 'cpu', 'cuda', 'disabled_deployment_input_names', 'disabled_deployment_output_names', 'double', 'dropout', 'dump_patches', 'dynamic_shapes_for_export', 'encoder_proj', 'eval', 'export', 'extra_repr', 'fc_d_model', 'fifo_len', 'first_hidden_to_hidden', 'float', 'forward', 'forward_speaker_sigm

In [13]:
# from omegaconf import OmegaConf
# import dataclasses
# from nemo.collections.asr.parts.mixins.diarization import DiarizeConfig

# print("model streaming attrs:", [a for a in dir(m) if "stream" in a.lower()])
# print("model forward attrs  :", [a for a in dir(m) if "forward" in a.lower()])
# print("---DiarizeConfig fields---")
# print([f.name for f in dataclasses.fields(DiarizeConfig)])
# print("---cfg keys---")
# print(list(m.cfg.keys()))
# for k in m.cfg:
#     if "stream" in k.lower() or "chunk" in k.lower():
#         print(k, "=", OmegaConf.to_container(m.cfg[k]) if hasattr(m.cfg[k], "keys") else m.cfg[k])


model streaming attrs: ['async_streaming', 'forward_streaming', 'forward_streaming_step', 'streaming_export', 'streaming_input_examples', 'streaming_mode']
model forward attrs  : ['_diarize_forward', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_slow_forward', 'forward', 'forward_for_export', 'forward_infer', 'forward_streaming', 'forward_streaming_step', 'register_forward_hook', 'register_forward_pre_hook']
---DiarizeConfig fields---
['session_len_sec', 'batch_size', 'num_workers', 'sample_rate', 'postprocessing_yaml', 'verbose', 'include_tensor_outputs', 'postprocessing_params', 'max_num_of_spks', '_internal']
---cfg keys---
['sample_rate', 'pil_weight', 'ats_weight', 'max_num_of_spks', 'model_defaults', 'train_ds', 'validation_ds', 'test_ds', 'preprocessor', 'sortformer_modules', 'encoder', 'transformer_encoder', 'loss', 'lr', 'optim', 'target', 'nemo_version']


In [9]:
import pathlib, subprocess
CODE = pathlib.Path("/kaggle/input/datasets/ritankarmondal/sarvam-diar-code")   # adjust if your slug differs
src = CODE / "upload_code" / "stage3_diarize.py"
print("source exists:", src.exists())
if src.exists():
    txt = src.read_text(encoding="utf-8")
    print("source has sortformer_stream:", txt.count("sortformer_stream"))
    print("source size:", src.stat().st_size)   # new file is 18463 bytes

source exists: True
source has sortformer_stream: 6
source size: 18463


In [10]:
!cp /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code/*.py /kaggle/working/
!grep -c sortformer_stream /kaggle/working/stage3_diarize.py

6


In [17]:
!grep -c sortformer_stream /kaggle/working/stage3_diarize.py

6


In [18]:
!python stage3_diarize.py --system sortformer_stream --data data --wav-dir {AUDIO} --limit 5

[env ] system=sortformer_stream  model=nvidia/diar_sortformer_4spk-v1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[warn] sortformer_stream is capped at 4 speakers; clips above that cannot be solved and should be reported separately
[run ] 5 of 5 clips to do (0 already done)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
diar_sortformer_4spk-v1.nemo: 100%|███████████| 493M/493M [00:03<00:00, 157MB/s]
[NeMo W 2026-09-07 17:27:13 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    sessi

In [ ]:
!python stage3_diarize.py --system sortformer_stream --data data --wav-dir {AUDIO}

In [11]:

import json, pathlib, collections
import pandas as pd

meta = pd.read_csv("/kaggle/working/data/ref/clip_meta.csv").set_index("clip_id")
pred = {}
for line in (pathlib.Path("/kaggle/working/data/hyp/sortformer_stream/manifest.jsonl")
             .read_text(encoding="utf-8").splitlines()):
    if line.strip():
        r = json.loads(line)
        if r["status"] == "ok":
            pred[r["clip_id"]] = r["n_speakers_pred"]

rows = [(int(meta.loc[c, "n_speakers"]), n) for c, n in pred.items() if c in meta.index]
ct = pd.crosstab(pd.Series([r for r, _ in rows], name="ref"),
                 pd.Series([p for _, p in rows], name="pred"))
print(ct)
exact = sum(r == p for r, p in rows)
print(f"\nexact match: {exact}/{len(rows)} = {exact/len(rows):.1%}")
print(f"exact match on ref<=4: "
      f"{sum(r == p for r, p in rows if r <= 4)}/{sum(1 for r, _ in rows if r <= 4)}")

pred   2   3   4
ref             
2     18   7   0
3      5  13  11
4      1   7  20
5      1   1   7
6      0   1   3
7      0   0   2
8      0   0   2

exact match: 51/99 = 51.5%
exact match on ref<=4: 51/82


In [12]:
!python stage3_score.py --data data --systems pyannote31 sortformer sortformer_stream --diagnostic


[ok] scored pyannote31: 99 clips
[ok] scored sortformer: 99 clips
[ok] scored sortformer_stream: 99 clips

STAGE 3 -- BASELINE DIARIZATION   (collar=0.0, skip_overlap=False, UEM=full clip)
system             DER    miss      FA    conf     JER  spk acc  spk MAE
------------------------------------------------------------------------------
pyannote31      27.34%  11.59%   5.89%   9.86%  38.14%    72.7%     0.33
sortformer      74.85%  65.14%   2.19%   7.52%  65.29%    43.4%     1.55
sortformer_stream  47.23%   9.82%   6.28%  31.13%  57.82%    51.5%     0.68
------------------------------------------------------------------------------
DER/miss/FA/conf are duration-weighted. Macro (per-clip mean) for contrast:
  pyannote31     DER_macro  29.76%   JER_macro  38.09%   (weighted DER 27.34%)
  sortformer     DER_macro  52.02%   JER_macro  59.57%   (weighted DER 74.85%)
    [!] 25 clip(s) had NO hypothesis (scored as total miss)
  sortformer_stream DER_macro  41.88%   JER_macro  53.95%   (wei

In [2]:

import pathlib
cands = sorted(pathlib.Path("/kaggle/input").glob("*/**/*.wav"))
print(f"{len(cands)} wavs found")
if cands:
    print("dir :", cands[0].parent)
    print("file:", cands[0].name)


99 wavs found
dir : /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
file: 0AEEA8NyVwY__000011000_000609000.wav


In [7]:
import pathlib
wav = str(sorted(pathlib.Path("/kaggle/input").glob("*/**/*.wav"))[0])
print("using:", wav)

import nemo.collections.asr.models as nam
names = [n for n in dir(nam) if "Model" in n]
print("has ASRModel:", "ASRModel" in names)
print("hybrid classes:", [n for n in names if "Hybrid" in n])

ASRModel = getattr(nam, "ASRModel")
MODEL_ID = "ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large"
print("model id len:", len(MODEL_ID))

using: /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav/0AEEA8NyVwY__000011000_000609000.wav
has ASRModel: True
hybrid classes: ['EncDecHybridRNNTCTCBPEModel', 'EncDecHybridRNNTCTCBPEModelWithPrompt', 'EncDecHybridRNNTCTCModel']
model id len: 53


In [12]:
!pip install -q "nemo_toolkit[asr]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 69.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pyannote-metrics 4.1 requires numpy>=2.2.2, but you have numpy 2.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [11]:
m = ASRModel.from_pretrained(MODEL_ID)
m.eval()

out = m.transcribe([wav], timestamps=True)
r = out[0]
print("type:", type(r))

ts = getattr(r, "timestamp", None)
print("timestamp keys:", list(ts.keys()) if isinstance(ts, dict) else type(ts))
if isinstance(ts, dict):
    for level in ("word", "segment", "char"):
        v = ts.get(level)
        print(f"  {level}: {len(v) if v is not None else None}")
        if v:
            print("   ", v[:3])

[NeMo I 2026-09-08 12:19:04 common:1210] Downloading ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large from HuggingFace Hub to path: /root/.cache/torch/NeMo/NeMo_3.0.0/hf_hub_cache/ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[NeMo I 2026-09-08 12:19:10 save_restore_connector:147] Restoration will occur within pre-extracted directory : `/root/.cache/torch/NeMo/NeMo_3.0.0/hf_hub_cache/ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8`.


FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/torch/NeMo/NeMo_3.0.0/hf_hub_cache/ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/model_config.yaml'

In [13]:
import pathlib
CACHE = pathlib.Path("/root/.cache/torch/NeMo/NeMo_3.0.0/hf_hub_cache")
for p in sorted(CACHE.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:8.1f} MB  {p.relative_to(CACHE)}")

     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/.cache/huggingface/.gitignore
     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/.cache/huggingface/CACHEDIR.TAG
     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/.cache/huggingface/download/.gitattributes.metadata
     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/.cache/huggingface/download/README.md.metadata
     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/.cache/huggingface/download/indicconformer_stt_hi_hybrid_rnnt_large.nemo.metadata
     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/.gitattributes
     0.0 MB  ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/README.md
   523.2 MB  ai4bharat/indicconf

In [14]:
nemo_files = sorted(CACHE.rglob("*.nemo"))
print("found:", nemo_files)

m = ASRModel.restore_from(str(nemo_files[0]), map_location="cpu")
m.eval()
print("class:", type(m).__name__)

found: [PosixPath('/root/.cache/torch/NeMo/NeMo_3.0.0/hf_hub_cache/ai4bharat/indicconformer_stt_hi_hybrid_ctc_rnnt_large/5e58728e282c92c6e67d1ac26cb8e5b8/indicconformer_stt_hi_hybrid_rnnt_large.nemo')]


[NeMo E 2026-09-08 12:25:06 common:827] Model instantiation failed!
    Target class:	nemo.collections.asr.models.hybrid_rnnt_ctc_bpe_models.EncDecHybridRNNTCTCBPEModel
    Error(s):	'dir'
    Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/nemo/core/classes/common.py", line 802, in from_config_dict
        instance = imported_cls(cfg=config, trainer=trainer)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      File "/usr/local/lib/python3.12/dist-packages/nemo/collections/asr/models/hybrid_rnnt_ctc_bpe_models.py", line 56, in __init__
        self._setup_tokenizer(cfg.tokenizer)
      File "/usr/local/lib/python3.12/dist-packages/nemo/collections/asr/parts/mixins/mixins.py", line 79, in _setup_tokenizer
        self._setup_monolingual_tokenizer(tokenizer_cfg)
      File "/usr/local/lib/python3.12/dist-packages/nemo/collections/asr/parts/mixins/mixins.py", line 89, in _setup_monolingual_tokenizer
        self.tokenizer_dir = self.tok

TypeError: Can't instantiate abstract class ASRModel without an implementation for abstract methods 'setup_training_data', 'setup_validation_data'

In [15]:
from omegaconf import OmegaConf
from nemo.collections.asr.models import EncDecHybridRNNTCTCBPEModel as HybridModel

P = str(nemo_files[0])
cfg = HybridModel.restore_from(P, return_config=True)
print(OmegaConf.to_yaml(cfg.tokenizer))
print("---- top-level keys ----")
print(list(cfg.keys()))

type: multilingual
langs:
  as:
    dir: /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/final_checkpoints/tokenizers/as_256/tokenizer_spe_bpe_v256
    type: bpe
    model_path: nemo:229442cc3c414cd1aa317112cab1b7a5_tokenizer.model
    vocab_path: nemo:2f99936bd7d743708c5988a05ba8b5aa_vocab.txt
    spe_tokenizer_vocab: nemo:33cdd8455569412eb79d224705398027_tokenizer.vocab
  bn:
    dir: /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/final_checkpoints/tokenizers/bn_256/tokenizer_spe_bpe_v256
    type: bpe
    model_path: nemo:8369760310ea4b97b171c511d26232f2_tokenizer.model
    vocab_path: nemo:e3063b78d108446caafdbb7e68f67235_vocab.txt
    spe_tokenizer_vocab: nemo:2933e76bb40048ae9499c2cdccfbd079_tokenizer.vocab
  brx:
    dir: /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/final_checkpoints/tokenizers/brx_256/tokenizer_spe_bpe_v256
    type: bpe
    model_path: nemo:c90ad90770f245d9b4671202c33adb17_tokenizer.model
    vocab_path: nemo:911bb6e9ec9

In [16]:
!sed -n '60,110p' /usr/local/lib/python3.12/dist-packages/nemo/collections/asr/parts/mixins/mixins.py


    # this will be used in configs and nemo artifacts
    AGGREGATE_TOKENIZERS_DICT_PREFIX = 'langs'

    @staticmethod
    def _get_extracted_tokenizer_name(member_name: str) -> str:
        member_file_name = PurePosixPath(member_name).name
        new_name = member_file_name.split("_")[1:]
        if len(new_name) > 1:
            return "_".join(new_name)
        return new_name[0]

    def _setup_tokenizer(self, tokenizer_cfg: DictConfig):
        tokenizer_type = tokenizer_cfg.get('type')
        if tokenizer_type is None:
            raise ValueError("`tokenizer.type` cannot be None")
        elif tokenizer_type.lower() == 'agg':
            self._setup_aggregate_tokenizer(tokenizer_cfg)
        else:
            self._setup_monolingual_tokenizer(tokenizer_cfg)

        self._derive_tokenizer_properties()

    def _setup_monolingual_tokenizer(self, tokenizer_cfg: DictConfig):
        # Prevent tokenizer parallelism (unless user has explicitly set it)
        if 'TOKENIZERS_PARA

In [17]:

cfg = HybridModel.restore_from(P, return_config=True)
print("before:", cfg.tokenizer.type)
cfg.tokenizer.type = "agg"

m = HybridModel.restore_from(P, override_config_path=cfg, map_location="cpu")
m.eval()
print("class :", type(m).__name__)
print("vocab :", m.tokenizer.vocab_size)
print("langs :", getattr(m.tokenizer, "langs", None))

before: multilingual
[NeMo I 2026-09-08 12:33:38 mixins:218] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:38 mixins:357] Tokenizer SentencePieceTokenizer initiali

[NeMo W 2026-09-08 12:33:44 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/manifests/nemo/vistaar_v3/train/train_hindi.json
    sample_rate: 16000
    batch_size: 8
    num_workers: 16
    pin_memory: true
    max_duration: 30.0
    min_duration: 0.2
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    is_concat: true
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1.5
    return_language_id: true
    
[NeMo W 2026-09-08 12:33:44 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the 

[NeMo I 2026-09-08 12:33:46 mixins:218] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-08 12:33:46 mixins:357] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[

[NeMo W 2026-09-08 12:33:51 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/manifests/nemo/vistaar_v3/train/train_hindi.json
    sample_rate: 16000
    batch_size: 8
    num_workers: 16
    pin_memory: true
    max_duration: 30.0
    min_duration: 0.2
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    is_concat: true
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1.5
    return_language_id: true
    
[NeMo W 2026-09-08 12:33:51 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the 

InstantiationException: Error in call to target 'nemo.collections.asr.modules.rnnt.RNNTDecoder':
TypeError("RNNTDecoder.__init__() got an unexpected keyword argument 'multisoftmax'")

In [18]:
print("decoder    multisoftmax:", cfg.decoder.get("multisoftmax"))
print("joint      multisoftmax:", cfg.joint.get("multisoftmax"))
print("aux_ctc decoder keys   :", list(cfg.aux_ctc.decoder.keys()))

decoder    multisoftmax: True
joint      multisoftmax: None
aux_ctc decoder keys   : ['_target_', 'feat_in', 'num_classes', 'vocabulary', 'multisoftmax']


In [19]:
!pip install -q onnxruntime
from huggingface_hub import list_repo_files
files = list_repo_files("ai4bharat/indic-conformer-600m-multilingual")
print([f for f in files if f.endswith(".onnx")])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 59.7 MB/s eta 0:00:00:00:0100:01
['assets/ctc_decoder.onnx', 'assets/encoder.onnx', 'assets/joint_enc.onnx', 'assets/joint_post_net_as.onnx', 'assets/joint_post_net_bn.onnx', 'assets/joint_post_net_brx.onnx', 'assets/joint_post_net_doi.onnx', 'assets/joint_post_net_gu.onnx', 'assets/joint_post_net_hi.onnx', 'assets/joint_post_net_kn.onnx', 'assets/joint_post_net_kok.onnx', 'assets/joint_post_net_ks.onnx', 'assets/joint_post_net_mai.onnx', 'assets/joint_post_net_ml.onnx', 'assets/joint_post_net_mni.onnx', 'assets/joint_post_net_mr.onnx', 'assets/joint_post_net_ne.onnx', 'assets/joint_post_net_or.onnx', 'assets/joint_post_net_pa.onnx', 'assets/joint_post_net_sa.onnx', 'assets/joint_post_net_sat.onnx', 'assets/joint_post_net_sd.onnx', 'assets/joint_post_net_ta.onnx', 'assets/joint_post_net_te.onnx', 'assets/joint_post_net_ur.onnx', 'assets/joint_pre_net.onnx', 'assets/joint_pred.onnx', 'assets/rnnt_decoder.onnx']


In [2]:
!pip install -q onnxruntime
from huggingface_hub import HfApi

REPO = "ai4bharat/indic-conformer-600m-multilingual"
info = HfApi().model_info(REPO, files_metadata=True)
for s in sorted(info.siblings, key=lambda x: -(x.size or 0)):
    print(f"{(s.size or 0)/1e6:9.1f} MB  {s.rfilename}")
    

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 71.7 MB/s eta 0:00:00:00:0100:01
     41.0 MB  assets/Constant_1970_attr__value
     40.7 MB  assets/rnnt_decoder.onnx
     23.1 MB  assets/ctc_decoder.onnx
     16.8 MB  assets/onnx__MatMul_8083
     16.8 MB  assets/onnx__MatMul_8084
     16.8 MB  assets/onnx__MatMul_8191
     16.8 MB  assets/onnx__MatMul_8192
     16.8 MB  assets/onnx__MatMul_8193
     16.8 MB  assets/onnx__MatMul_8194
     16.8 MB  assets/onnx__MatMul_8229
     16.8 MB  assets/onnx__MatMul_8230
     16.8 MB  assets/onnx__MatMul_8231
     16.8 MB  assets/onnx__MatMul_8232
     16.8 MB  assets/onnx__MatMul_8267
     16.8 MB  assets/onnx__MatMul_8268
     16.8 MB  assets/onnx__MatMul_8269
     16.8 MB  assets/onnx__MatMul_8270
     16.8 MB  assets/onnx__MatMul_8305
     16.8 MB  assets/onnx__MatMul_8306
     16.8 MB  assets/onnx__MatMul_8307
     16.8 MB  assets/onnx__MatMul_8308
     16.8 MB  assets/onnx__MatMul_8343
     16.8 MB  assets/onnx__MatMul_8344
     1

In [5]:
onnx = [(s.rfilename, s.size or 0) for s in info.siblings if s.rfilename.endswith(".onnx")]
for name, size in sorted(onnx, key=lambda x: -x[1]):
    print(f"{size/1e6:9.2f} MB  {name}")

    40.68 MB  assets/rnnt_decoder.onnx
    23.10 MB  assets/ctc_decoder.onnx
     2.98 MB  assets/encoder.onnx
     2.63 MB  assets/joint_enc.onnx
     1.65 MB  assets/joint_pred.onnx
     0.66 MB  assets/joint_post_net_as.onnx
     0.66 MB  assets/joint_post_net_bn.onnx
     0.66 MB  assets/joint_post_net_brx.onnx
     0.66 MB  assets/joint_post_net_doi.onnx
     0.66 MB  assets/joint_post_net_gu.onnx
     0.66 MB  assets/joint_post_net_hi.onnx
     0.66 MB  assets/joint_post_net_kn.onnx
     0.66 MB  assets/joint_post_net_kok.onnx
     0.66 MB  assets/joint_post_net_ks.onnx
     0.66 MB  assets/joint_post_net_mai.onnx
     0.66 MB  assets/joint_post_net_ml.onnx
     0.66 MB  assets/joint_post_net_mni.onnx
     0.66 MB  assets/joint_post_net_mr.onnx
     0.66 MB  assets/joint_post_net_ne.onnx
     0.66 MB  assets/joint_post_net_or.onnx
     0.66 MB  assets/joint_post_net_pa.onnx
     0.66 MB  assets/joint_post_net_sa.onnx
     0.66 MB  assets/joint_post_net_sat.onnx
     0.66 MB  asse

In [6]:
other = [(s.rfilename, s.size or 0) for s in info.siblings
         if not s.rfilename.endswith(".onnx")]
for name, size in sorted(other, key=lambda x: -x[1]):
    print(f"{size/1e6:9.2f} MB  {name}")

    40.96 MB  assets/Constant_1970_attr__value
    16.78 MB  assets/onnx__MatMul_8083
    16.78 MB  assets/onnx__MatMul_8084
    16.78 MB  assets/onnx__MatMul_8191
    16.78 MB  assets/onnx__MatMul_8192
    16.78 MB  assets/onnx__MatMul_8193
    16.78 MB  assets/onnx__MatMul_8194
    16.78 MB  assets/onnx__MatMul_8229
    16.78 MB  assets/onnx__MatMul_8230
    16.78 MB  assets/onnx__MatMul_8231
    16.78 MB  assets/onnx__MatMul_8232
    16.78 MB  assets/onnx__MatMul_8267
    16.78 MB  assets/onnx__MatMul_8268
    16.78 MB  assets/onnx__MatMul_8269
    16.78 MB  assets/onnx__MatMul_8270
    16.78 MB  assets/onnx__MatMul_8305
    16.78 MB  assets/onnx__MatMul_8306
    16.78 MB  assets/onnx__MatMul_8307
    16.78 MB  assets/onnx__MatMul_8308
    16.78 MB  assets/onnx__MatMul_8343
    16.78 MB  assets/onnx__MatMul_8344
    16.78 MB  assets/onnx__MatMul_8345
    16.78 MB  assets/onnx__MatMul_8346
    16.78 MB  assets/onnx__MatMul_8381
    16.78 MB  assets/onnx__MatMul_8382
    16.78 MB  ass

In [4]:
from huggingface_hub import hf_hub_download
import onnxruntime as ort

FILE = "assets/joint_pre_net.onnx"
path = hf_hub_download(REPO, FILE)
sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])

print("INPUTS")
for i in sess.get_inputs():
    print(" ", i.name, i.shape, i.type)
print("OUTPUTS")
for o in sess.get_outputs():
    print(" ", o.name, o.shape, o.type)

assets/joint_pre_net.onnx:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

INPUTS
  input ['B', 'T', 640] tensor(float)
OUTPUTS
  output ['B', 'T', 640] tensor(float)


In [7]:
from huggingface_hub import snapshot_download
import onnxruntime as ort, json, torch

LOCAL = snapshot_download(REPO, allow_patterns=["assets/*", "*.json"])
print("local:", LOCAL)

enc = ort.InferenceSession(f"{LOCAL}/assets/encoder.onnx", providers=["CPUExecutionProvider"])
ctc = ort.InferenceSession(f"{LOCAL}/assets/ctc_decoder.onnx", providers=["CPUExecutionProvider"])

for tag, s in (("ENCODER", enc), ("CTC", ctc)):
    print(tag)
    for i in s.get_inputs():
        print("   in :", i.name, i.shape, i.type)
    for o in s.get_outputs():
        print("   out:", o.name, o.shape, o.type)

vocab = json.load(open(f"{LOCAL}/assets/vocab.json"))
print("vocab type:", type(vocab), "len:", len(vocab))
print("first 10:", list(vocab)[:10] if isinstance(vocab, dict) else vocab[:10])

pre = torch.jit.load(f"{LOCAL}/assets/preprocessor.ts")
print("preprocessor:", pre.forward.schema)


Fetching 399 files:   0%|          | 0/399 [00:00<?, ?it/s]

local: /root/.cache/huggingface/hub/models--ai4bharat--indic-conformer-600m-multilingual/snapshots/e9b71b369c048e2c6b634d4c131061c34e441179
ENCODER
   in : audio_signal ['audio_signal_dynamic_axes_1', 80, 'audio_signal_dynamic_axes_2'] tensor(float)
   in : length ['length_dynamic_axes_1'] tensor(int64)
   out: outputs ['outputs_dynamic_axes_1', 1024, 'outputs_dynamic_axes_2'] tensor(float)
   out: encoded_lengths ['encoded_lengths_dynamic_axes_1'] tensor(int64)
CTC
   in : encoder_output ['encoder_output_dynamic_axes_1', 1024, 'encoder_output_dynamic_axes_2'] tensor(float)
   out: logprobs ['logprobs_dynamic_axes_1', 'logprobs_dynamic_axes_2', 5633] tensor(float)
vocab type: <class 'dict'> len: 22
first 10: ['as', 'bn', 'brx', 'doi', 'kok', 'gu', 'hi', 'kn', 'ks', 'mai']
preprocessor: forward(__torch__.nemo.collections.asr.modules.audio_preprocessing.___torch_mangle_0.AudioToMelSpectrogramPreprocessor self, Tensor input_signal, Tensor length) -> ((Tensor, Tensor))
